<a href="https://colab.research.google.com/github/Reileen00/OpenRLHF-PPO/blob/main/OpenRLHF_PPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import wandb
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: baisnabirout001 (baisnabirout001-iit-kharagpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
!pip install --upgrade datasets
!pip install openrlhf
!pip install openrlhf[vllm]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 34.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 98.0 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
  Using cached openrlhf-0.9.6-cp312-cp312-manylinux1_x86_64.whl.metadata (38 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached deepspeed-0.18.8.tar.gz (1.6 MB)
  Preparing metadata (setup.py) ... done
  Using cached flash_attn-2.8.3.tar.gz (8.4 MB)
  error: su

In [3]:
## Prepare the Dataset

from datasets import load_dataset
import os

N_TRAIN = 1600
N_TEST = 160
SEED = 42

#Load and shuffle the dataset
full = load_dataset("OpenRLHF/prompt-collection-v0.1",split="train")
full = full.shuffle(seed=SEED)

#Split into training and testing subsets
train_dataset = full.select(range(N_TRAIN))
test_dataset = full.select(range(N_TRAIN,N_TRAIN + N_TEST))

#Save to disk
train_dataset.save_to_disk("dataset/train")
test_dataset.save_to_disk("dataset/test")

print(f"Saved {len(train_dataset)} train examples")
print(f"Saved {len(test_dataset)} test examples")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Saving the dataset (0/1 shards):   0%|          | 0/1600 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/160 [00:00<?, ? examples/s]

Saved 1600 train examples
Saved 160 test examples


In [4]:
#Prompt Examples

from datasets import load_from_disk

# Load the training dataset from disk
train_dataset_path = "dataset/train"
train_dataset = load_from_disk(train_dataset_path)

#Select and display an example prompt
if len(train_dataset) > 0:
  example_prompt = train_dataset[2]
  print("Example Prompt:")
  print(example_prompt)
else:
  print("The training dataset is empty.")

Example Prompt:
{'dataset': 'OpenOrca', 'context': 'Here is a request of a user for an AI assistant.\n\nUser:\nAnswer the following question given this paragraph:   Paleontology, another branch of biology, uses fossils to study life’s history (Figure 1.20). Zoology and botany are the study of animals and plants, respectively. Biologists can also specialize as biotechnologists, ecologists, or physiologists, to name just a few areas. This is just a small sample of the many fields that biologists can pursue. Biology is the culmination of the achievements of the natural sciences from their inception to today. Excitingly, it is the cradle of emerging sciences, such as the biology of brain activity, genetic engineering of custom organisms, and the biology of evolution that uses the laboratory tools of molecular biology to retrace the earliest stages of life on earth. A scan of news headlines—whether reporting on immunizations, a newly discovered species, sports doping, or a genetically-modif

In [5]:
#Installation of Flash Attention
!pip uninstall -y flash-attn
!pip cache purge
!pip install flash-attn --no-cache-dir


Files removed: 18
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 85.9 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [6]:
!pip install --upgrade setuptools wheel
!pip uninstall -y flash-attn
!pip cache purge
!FLASH_ATTENTION_NO_CUDA_EXTENSIONS=1 pip install flash-attn==2.8.3 --no-build-isolation
!pip install openrlhf
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer
from openrlhf.models import get_llm_for_sequence_regression
from vllm import LLM, SamplingParams
from tqdm.notebook import tqdm

def evaluate_model_vllm(test_dataset_path, rm_name, lm_name, seed=42, max_new_tokens = 1024):
  # Load your on-disk test dataset
  ds = load_from_disk(test_dataset_path)
  raw_prompts = ds['context_messages']
  num_samples = len(raw_prompts)

  # Prepare reward-model components
  rm_tokenizer = AutoTokenizer.from_pretrained(rm_name)
  rm_model = get_llm_for_sequence_regression(
      model_name_or_path = rm_name,
      model_type = "reward",
      bf16 = True,
      init_value_head = False,
  )
  rm_model.eval().to("cuda")

  # Prepare vLLM engine for generation
  lm_tokenizer = AutoTokenizer.from_pretrained(lm_name)
  llm_engine = LLM(
      model = lm_name,
      enforce_eager = True,
      seed = seed,
      tensor_parallel_size = 1,
      trust_remote_code = True,
      enable_sleep_mode = True,
      gpu_memory_utilization = 0.8,
      dtype = "bfloat16"
  )

  # Build formatted prompts for vLLM
  formatted_prompts = []
  for prompt in raw_prompts:
    # Apply the same chat template you used before
    prompt_text = lm_tokenizer.apply_chat_template(
        prompt, tokenize = False, add_generation_prompt = True
    )
    formatted_prompts.append(prompt_text)

  # Sampling settings
  sampling_params = SamplingParams(
      max_tokens = max_new_tokens,
      min_tokens = 1,
      temperature = 1.0,
      top_p = 1.0,
      top_k = 1, # only the highest probability token is sampled
      seed = seed,
      skip_special_tokens = False,
      include_stop_str_in_output = True,
  )
  # Run batched inference
  outputs = llm_engine.generate(formatted_prompts, sampling_params, use_tqdm = True)
  llm_engine.sleep(level=2) # Discards both the model weights and the KV cache

  total_reward = 0.0
  total_length = 0
  evaluation_results = [] # Initialize list to store results

  progressBar = tqdm(total = num_samples, desc = "Evaluate Rewards")
  # Iterate through vLLM outputs in the same order as your prompts
  for i, output in enumerate(outputs):
    raw_prompt = raw_prompts[i]
    prompt = rm_tokenizer.apply_chat_template(
        raw_prompt, tokenize = False, add_generation_prompt = True
    )
    response = output.outputs[0].text
    # Concatenate original raw prompt + response for the RM input

    rm_input = prompt + response

    # Score with the reward model
    rm_input = rm_tokenizer(rm_input, return_tensors = "pt", truncation = True, padding= True).to(rm_model.device)
    with torch.inference_mode():
      reward_score = rm_model(**rm_input)
    reward_score = float(reward_score)
    total_reward += reward_score

    response_ids = rm_tokenizer(response , return_tensors = "pt", truncation = True, padding = True).to(rm_model.device)
    total_length += response_ids['input_ids'].shape[1]

    # Store the results
    evaluation_results.append(
      {
          'prompt': raw_prompt,
          'response': response,
          'reward': reward_score
      }
    )

    progressBar.update(1)

  progressBar.close()

  avg_reward = total_reward / num_samples
  avg_length = total_length / num_samples

  print(f"Samples evaluated: {num_samples}")
  print(f"Average reward: {avg_reward:.4f}")
  print(f"Average response length (tokens): {avg_length:.2f}")

  return evaluation_results #Return the results

# The call to the function needs to be updated to capture the returned value
# results = evaluate_model_vllm(...)
# This will be done in the next step as the subtask is only to modify the function.


Files removed: 4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 62.1 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 36.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached flash_attn-2.8.3.tar.gz (8.4 MB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata

ModuleNotFoundError: No module named 'openrlhf'

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
if 'evaluation_results' in locals() and evaluation_results:
  # Convert to DataFrame for easier handling
  results_df = pd.DataFrame(evaluation_results)

  # Sort by reward to find high and low examples
  results_df_sorted = results_df.sort_values(by = 'reward')

  # Get example with lowest reward
  lowest_reward_example = results_df_sorted.iloc[0]
  print("\n--- Example with Lowest Reward ---")
  print(f"Prompt: {lowest_reward_example['prompt']}")
  print(f"Response: {lowest_reward_example['response']}")
  print(f"Reward: {lowest_reward_example['reward']:.4f}")

  # Get example with highest reward
  highest_reward_example = results_df_sorted_iloc[-1]
  print("\n--- Example with Highest Reward ---")
  print(f"Prompt: {highest_reward_example_reward_example['prompt']}")
  print(f"Response: {highest_reward_example['response']}")
  print(f"Reward: {highest_reward_example['reward']:.4f}")

  # Plot the distribution of rewards
  plt.figure(figsize=(10,6))
  plt.hist(results_df['reward'],bins=20,edgecolor='black')
  plt.xlabel("Reward")
  plt.ylabel("Frequency")
  plt.title("Distribution of Rewards")
  plt.grid(True)
  plt.show()

else:
  print("No evaluation results available to display or plot. Please run the evaluation cell first.")

No evaluation results available to display or plot. Please run the evaluation cell first.


In [3]:
import os, json, subprocess, re
import matplotlib.pyplot as plt

# --- WANDB config ---
WANDB_PROJECT = "OpenRLHF_ppo_train"

# --- Ray working dir and runtime env ---
working_dir = "openrlhf"
os.makedirs(working_dir, exist_ok=True)

runtime_env = {
    "working_dir": f"./{working_dir}",
    "env_vars": {
        "WANDB_PROJECT": WANDB_PROJECT,
        "WANDB_MODE": "online",
    },
}
runtime_env_json = json.dumps(runtime_env)

# --- Start Ray head (idempotent) ---
try:
    print("Starting Ray head node...")
    subprocess.run(["ray", "start", "--head", "--node-ip-address", "0.0.0.0", "--num-gpus", "1"], check=True)
except subprocess.CalledProcessError:
    print("Ray may already be running, skipping Ray start.")

# --- Submit PPO job ---
submit_ray_job_command = [
    "ray", "job", "submit", "--address=http://127.0.0.1:8265",
    f"--runtime-env-json={runtime_env_json}",
    "--", "python3", "-m", "openrlhf.cli.train_ppo_ray",
    "--ref_num_gpus_per_node", "1",
    "--reward_num_gpus_per_node", "1",
    "--critic_num_gpus_per_node", "1",
    "--actor_num_gpus_per_node", "1",
    "--vllm_num_engines", "1",
    "--pretrain", "HuggingFaceTB/SmolLM2-135M-Instruct",
    "--reward_pretrain", "AI-Roadmap/SmolLM2-135M-rm-60k",
    "--save_path", "/content/final/SmolLM2-135M-rlhf",
    "--ckpt_path", "/content/ckpt/SmolLM2-135M-rlhf",
    "--save_hf_ckpt",
    "--save_steps", "10",
    "--micro_train_batch_size", "8",
    "--micro_rollout_batch_size", "8",
    "--rollout_batch_size", "32",
    "--prompt_max_len", "1024",
    "--generate_max_len", "1024",
    "--zero_stage", "0",
    "--actor_learning_rate", "5e-7",
    "--critic_learning_rate", "9e-6",
    "--init_kl_coef", "0.01",
    "--prompt_data", "/content/dataset/train",
    "--vllm_gpu_memory_utilization", "0.7",
    "--input_key", "context_messages",
    "--apply_chat_template",
    "--normalize_reward",
    "--gradient_checkpointing",
    "--packing_samples",
    "--vllm_sync_backend", "nccl",
    "--vllm_enable_sleep",
    "--colocate_all_models",
    "--bf16",
    "--enforce_eager",
    "--use_kl_loss",
    "--save_value_network",

    # --- W&B parameters (simplified) ---
    "--use_wandb", "Your API Key",
    "--wandb_project", "OpenRLHF_ppo_train",
    "--wandb_run_name", "SmolLM2-135M-PPO-training"
]

print("\nSubmitting Ray job...")
proc = subprocess.Popen(submit_ray_job_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()

Starting Ray head node...


FileNotFoundError: [Errno 2] No such file or directory: 'ray'

In [4]:
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer
from openrlhf.models import get_llm_for_sequence_regression
from vllm import LLM, SamplingParams
from tqdm.notebook import tqdm

def evaluate_model_vllm(test_dataset_path, rm_name, lm_repo, seed=42, max_new_tokens=1024):
    # Load your on-disk test dataset
    ds = load_from_disk(test_dataset_path)
    raw_prompts = ds['context_messages']
    num_samples = len(raw_prompts)

    # Prepare reward‐model components

    rm_tokenizer = AutoTokenizer.from_pretrained(rm_name)
    rm_model = get_llm_for_sequence_regression(
        model_name_or_path=rm_name,
        model_type="reward",
        bf16=True,
        init_value_head=False,
    )
    rm_model.eval().to("cuda")

    # Prepare vLLM engine for generation
    lm_tokenizer = AutoTokenizer.from_pretrained(lm_repo)
    llm_engine = LLM(
        model=lm_repo,
        enforce_eager=True,
        seed=seed,
        tensor_parallel_size=1,
        trust_remote_code=True,
        enable_sleep_mode=True,
        gpu_memory_utilization=0.8,
        dtype="bfloat16",
    )

    # Build formatted prompts for vLLM
    formatted_prompts = []
    for prompt in raw_prompts:
        # Apply the same chat template you used before
        prompt_text = lm_tokenizer.apply_chat_template(
            prompt, tokenize=False, add_generation_prompt=True
        )
        formatted_prompts.append(prompt_text)

    # Sampling settings
    sampling_params = SamplingParams(
        max_tokens=max_new_tokens,
        min_tokens=1,
        temperature=1.0,
        top_p=1.0,
        top_k=1,  # only the highest-probability token is sampled
        seed=seed,
        skip_special_tokens=False,
        include_stop_str_in_output=True,
    )
    # Run batched inference
    outputs = llm_engine.generate(formatted_prompts, sampling_params, use_tqdm=True)
    llm_engine.sleep(level=2) # Discards both the model weights and the KV cache

    total_reward = 0.0
    total_length = 0
    evaluation_results = [] # Initialize list to store results

    progressBar = tqdm(total=num_samples, desc="Evaluate Rewars")
    # Iterate through vLLM outputs in the same order as your prompts
    for i, output in enumerate(outputs):
        raw_prompt = raw_prompts[i]
        prompt = rm_tokenizer.apply_chat_template(
            raw_prompt, tokenize=False, add_generation_prompt=True
        )
        response = output.outputs[0].text
        # Concatenate original raw prompt + response for the RM input

        rm_input = prompt + response

        # Score with the reward model
        rm_input = rm_tokenizer(rm_input, return_tensors="pt", truncation=True, padding=True).to(rm_model.device)
        with torch.inference_mode():
          reward_score = rm_model(**rm_input)
        reward_score = float(reward_score)
        total_reward += reward_score

        response_ids = rm_tokenizer(response, return_tensors="pt", truncation=True, padding=True).to(rm_model.device)
        total_length += response_ids['input_ids'].shape[1]

        # Store the results
        evaluation_results.append({
            'prompt': raw_prompt,
            'response': response,
            'reward': reward_score
        })

        progressBar.update(1)

    progressBar.close()

    avg_reward = total_reward / num_samples
    avg_length = total_length / num_samples

    print(f"Samples evaluated: {num_samples}")
    print(f"Average reward: {avg_reward:.4f}")
    print(f"Average response length (tokens): {avg_length:.2f}")

    return evaluation_results # Return the results

# The call to the function needs to be updated to capture the returned value
# results = evaluate_model_vllm(...)
# This will be done in the next step as the subtask is only to modify the function.

ModuleNotFoundError: No module named 'openrlhf.models'

In [5]:
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer
from openrlhf.models import get_llm_for_sequence_regression
from vllm import LLM, SamplingParams
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import pandas as pd

# Assuming evaluation_results from the SFT model evaluation is available from a previous cell run.
# For comparison, let's store the SFT results in a variable if they are not already.
# If you haven't run the SFT evaluation and stored its results, you'll need to do that first.

# Assuming 'evaluation_results' from the SFT model evaluation is in the kernel's namespace.
# If not, you might need to load it from wherever it was stored or re-run the SFT evaluation.
if 'evaluation_results' in locals():
    sft_evaluation_results = evaluation_results
    sft_avg_reward = sum([res['reward'] for res in sft_evaluation_results]) / len(sft_evaluation_results)
    print(f"Average reward for SFT model: {sft_avg_reward:.4f}")
else:
    print("SFT evaluation results not found. Please run the SFT model evaluation first.")
    sft_avg_reward = None # Set to None if SFT results are not available


# Evaluate the RLHF model
print("\nEvaluating RLHF model...")
rlhf_evaluation_results = evaluate_model_vllm("dataset/test",
                                             rm_name="AI-Roadmap/SmolLM2-135M-rm-60k",
                                             lm_repo="/content/final/SmolLM2-135M-rlhf", # Path to your trained RLHF model
                                             max_new_tokens=1024)

# Calculate average reward for RLHF model
rlhf_avg_reward = sum([res['reward'] for res in rlhf_evaluation_results]) / len(rlhf_evaluation_results)
print(f"Average reward for RLHF model: {rlhf_avg_reward:.4f}")

# Compare the average rewards
if sft_avg_reward is not None:
    print("\n--- Reward Comparison (Average) ---")
    print(f"SFT Model Average Reward: {sft_avg_reward:.4f}")
    print(f"RLHF Model Average Reward: {rlhf_avg_reward:.4f}")

    if rlhf_avg_reward > sft_avg_reward:
        print("The RLHF model achieved a higher average reward than the SFT model.")
    elif rlhf_avg_reward < sft_avg_reward:
        print("The RLHF model achieved a lower average reward than the SFT model.")
    else:
        print("The RLHF and SFT models achieved the same average reward.")
else:
    print("\nCannot compare average rewards as SFT evaluation results were not available.")

# Plotting the distribution of rewards for comparison
if sft_evaluation_results is not None and rlhf_evaluation_results:
    sft_rewards = [res['reward'] for res in sft_evaluation_results]
    rlhf_rewards = [res['reward'] for res in rlhf_evaluation_results]

    plt.figure(figsize=(12, 7))

    plt.hist(sft_rewards, bins=20, alpha=0.7, label='SFT Model', edgecolor='black')
    plt.hist(rlhf_rewards, bins=20, alpha=0.7, label='RLHF Model', edgecolor='black')

    plt.xlabel("Reward")
    plt.ylabel("Frequency")
    plt.title("Distribution of Rewards: SFT vs. RLHF Model")
    plt.legend()
    plt.grid(True)
    plt.show()
elif sft_evaluation_results is None and rlhf_evaluation_results:
    print("\nOnly RLHF model evaluation results are available. Plotting RLHF reward distribution.")
    rlhf_rewards = [res['reward'] for res in rlhf_evaluation_results]
    plt.figure(figsize=(10, 6))
    plt.hist(rlhf_rewards, bins=20, edgecolor='black')
    plt.xlabel("Reward")
    plt.ylabel("Frequency")
    plt.title("Distribution of Rewards: RLHF Model")
    plt.grid(True)
    plt.show()
else:
    print("\nNo evaluation results available to plot distributions.")

ModuleNotFoundError: No module named 'openrlhf.models'